In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
num_epochs = 4
batch_size = 4
learning_rate = 0.001

In [4]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [5]:
train_dataset = torchvision.datasets.CIFAR10(root='./data', download=True, train=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='./data', download=False, train=False, transform=transform)

train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [6]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [7]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 16*5*5)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
model = ConvNet().to(device)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [9]:
n_total_steps = len(train_dataloader)

for epoch in range(num_epochs):
    for i, (images,labels) in enumerate(train_dataloader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 2000 == 0:
            print(f"epoch [{epoch+1}/{num_epochs}], step [{i+1}/{n_total_steps}], loss: {loss.item():.4f}")

epoch [1/4], step [2000/12500], loss: 1.0165
epoch [1/4], step [4000/12500], loss: 0.7970
epoch [1/4], step [6000/12500], loss: 1.2624
epoch [1/4], step [8000/12500], loss: 1.5721
epoch [1/4], step [10000/12500], loss: 1.2315
epoch [1/4], step [12000/12500], loss: 2.0525
epoch [2/4], step [2000/12500], loss: 1.8164
epoch [2/4], step [4000/12500], loss: 0.8596
epoch [2/4], step [6000/12500], loss: 0.7765
epoch [2/4], step [8000/12500], loss: 0.4573
epoch [2/4], step [10000/12500], loss: 1.3785
epoch [2/4], step [12000/12500], loss: 1.3777
epoch [3/4], step [2000/12500], loss: 1.0385
epoch [3/4], step [4000/12500], loss: 1.4853
epoch [3/4], step [6000/12500], loss: 0.7764
epoch [3/4], step [8000/12500], loss: 1.4989
epoch [3/4], step [10000/12500], loss: 2.3545
epoch [3/4], step [12000/12500], loss: 1.2628
epoch [4/4], step [2000/12500], loss: 0.4516
epoch [4/4], step [4000/12500], loss: 1.5203
epoch [4/4], step [6000/12500], loss: 0.1455
epoch [4/4], step [8000/12500], loss: 0.9996
epoc

In [10]:
with torch.no_grad():
    n_correct = 0
    n_samples = 0
    n_class_correct = [0 for i in range(10)]
    n_class_samples = [0 for i in range(10)]
    for images, labels in test_dataloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        
        _, predictions = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predictions == labels).sum().item()
        
        acc = 100.0 * n_correct / n_samples
    
        for i in range(len(images)):
            label = labels[i]
            pred = predictions[i]
            if (label == pred):
                n_class_correct[label] += 1
            n_class_samples[label] += 1
            
    print(f"Accuracy of the network: {acc} %")
    
    for i in range(10):
        acc = 100.0 * n_class_correct[i] / n_class_samples[i]
        print(f"Accuracy of {classes[i]}: {acc} %")
    

Accuracy of the network: 58.68 %
Accuracy of plane: 61.9 %
Accuracy of car: 80.6 %
Accuracy of bird: 43.2 %
Accuracy of cat: 26.4 %
Accuracy of deer: 49.1 %
Accuracy of dog: 62.0 %
Accuracy of frog: 75.0 %
Accuracy of horse: 66.7 %
Accuracy of ship: 63.5 %
Accuracy of truck: 58.4 %
